<a href="https://colab.research.google.com/github/mbardela/ementas_tit/blob/main/Clusterizacao_TIT_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Preparação/importação



##pip


In [ ]:
#!pip install numpy==1.26.4 pandas==2.2.2 cupy-cuda12x>=13.6.0 --force-reinstall
#!pip install -U spacy[cuda12x]
!pip install spacy
!pip install transformers
!pip install kneed
!pip install wordcloud


!python -m spacy download pt_core_news_lg

##import

In [ ]:
import pandas as pd
import numpy as np
import re
import cudf
import joblib
import nltk
import seaborn as sns
import matplotlib.pyplot as plt
import spacy

from cuml.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer
from cuml.decomposition import TruncatedSVD
from cuml.preprocessing import Normalizer, StandardScaler, MinMaxScaler, RobustScaler
from cuml.pipeline import make_pipeline
from cuml.cluster import KMeans

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

from transformers import AutoTokenizer, AutoModel
from kneed import KneeLocator
from wordcloud import WordCloud


#Constrói dataframe principal

In [ ]:
#importa CSV do scrap
decisoes_df = pd.DataFrame()
arquivo = 'https://github.com/mbardela/ementas_tit/raw/refs/heads/main/ementas_tit.csv'
decisoes_df = pd.read_csv(arquivo, sep=';', encoding='utf8')

#renomeia algumas colunas do scrap e ajusta dtype, etc
decisoes_df.rename(columns={'Ementa': 'ementa'}, inplace=True)
decisoes_df.rename(columns={'Data da Publicação': 'data_pub'}, inplace=True)
decisoes_df['data_pub'] = pd.to_datetime(decisoes_df['data_pub'], format='%d/%m/%Y')
decisoes_df['ano_pub'] = decisoes_df['data_pub'].dt.year

# Extrai os 7 primeiros dígitos da coluna 'AIIM' e converte para integer
decisoes_df['AIIM_int'] = pd.to_numeric(decisoes_df['AIIM'].astype(str).str.replace(r'\D', '', regex=True).str[:7], errors='coerce').fillna(0).astype(int)

#dropNA
decisoes_df.dropna(inplace=True)

#LIMPA QUEBRA DE LINHA, TAB, ETC
def clean_text(text):
    if pd.isna(text):
        return ''
    text = text.replace('\t', ' ')
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


decisoes_df['ementa'] = decisoes_df['ementa'].apply(clean_text)


In [ ]:
# Função para criar ementa_clean (limpeza de regex)
def clean_and_filter(texto):
    # Divide o texto em palavras
    itens = texto.split(' ')
    palavras_filtradas = []

    for palavra in itens:
        # Mantém apenas os caracteres abaixo usando regex
        palavra_limpa = re.sub(r'[^a-zA-ZáàâãéèêíïóôõöúçñÁÀÂÃÉÈÊÍÏÓÔÕÖÚÇÑ0123456789]', '', palavra).lower()

        # Filtra se não estiver vazia ()
        if palavra_limpa.strip():
            palavras_filtradas.append(palavra_limpa)

    return ' '.join(palavras_filtradas)



# Cria a coluna ementa_clean primeiro
decisoes_df['ementa_clean'] = decisoes_df['ementa'].apply(clean_and_filter)


## Ajusta ementas (poda extremos)

In [ ]:
# Adiciona contagem de palavras da coluna "ementa_clean" em decisoes_df na coluna 'ementa_word_count'

decisoes_df['ementa_word_count'] = decisoes_df['ementa_clean'].apply(lambda x: len(x.split(' ')))

# Filtragem
# exploração iterativa
# parece que <10 é só diligência ou lixo

i=10
ementas_filtrado = decisoes_df[decisoes_df['ementa_word_count'] < i]

#display(ementas_filtrado)





In [ ]:
# da exploração do dataset acima, temos um número grande diligências. iremos expurgar
# Lista de termos para exclusão
termos_excluir = ['conversão do julgamento em diligência', 'convertido em diligência', 'conversão em diligência']

# Cria um padrão regex unindo os termos com o operador OR (|)
padrao_exclusao = '|'.join(termos_excluir)

# Filtra o dataframe mantendo apenas o que não contém os termos
decisoes_df = decisoes_df[~decisoes_df['ementa'].str.contains(padrao_exclusao, case=False, na=False)]

# Verifica o resultado

#display(ementas_filtrado)

In [ ]:
#histograma de contagem de palavras em 'ementa'

max_word_count = decisoes_df['ementa_word_count'].max()
bins = np.arange(0, max_word_count + 100, 100)

fig, ax = plt.subplots(figsize=(16, 6))
sns.histplot(decisoes_df['ementa_word_count'], bins=bins, kde=False)
plt.title('Distribuição do tamanho da ementa (qtde de palavras)', fontsize=16, fontweight='bold')
ax.set_xlabel('Quantidade de palavras', fontsize=12, fontweight='bold')
ax.set_ylabel('Qtde. de ementas', fontsize=12, fontweight='bold')
plt.xticks(bins[::5])
plt.show()

In [ ]:
# gráfico de distribuição long tail
# analisar extremos superiores

# Define os limites dos bins
i_bin =50
max_words = decisoes_df['ementa_word_count'].max()
bins = list(range(0, int(max_words) + i_bin, i_bin))

# Cria as categorias e conta as ocorrências
decisoes_df['bins_texto'] = pd.cut(decisoes_df['ementa_word_count'], bins=bins)
contagem_bins = decisoes_df['bins_texto'].value_counts().sort_index().reset_index()
contagem_bins.columns = ['Intervalo (Qtd. Palavras)', 'Total de Ementas']

# Calcula o total geral de ementas
total_geral_ementas = contagem_bins['Total de Ementas'].sum()

# Adiciona a coluna com o percentual do total
contagem_bins['% do Total'] = (contagem_bins['Total de Ementas'] / total_geral_ementas * 100).round(2)

# Exibe o resultado formatado como texto/tabela
display(contagem_bins)

In [ ]:
#repete o histograma, agora mais detalhado
i_bin =50
max_word_count = decisoes_df['ementa_word_count'].max()
bins = np.arange(0, max_word_count + i_bin, i_bin)

fig, ax = plt.subplots(figsize=(16, 6))
hist_plot = sns.histplot(decisoes_df['ementa_word_count'], bins=bins, kde=False, ax=ax)
plt.title('Distribuição do tamanho da ementa (qtde de palavras)', fontsize=16, fontweight='bold')
ax.set_xlabel('Quantidade de palavras', fontsize=12, fontweight='bold')
ax.set_ylabel('Qtde. de ementas', fontsize=12, fontweight='bold')
plt.xticks(bins[::1])

# Adiciona o percentual acumulado em cada barra
total_ementas = len(decisoes_df)

# Get the counts for each bin from the hist_plot.patches
counts = [p.get_height() for p in hist_plot.patches]
cumulative_counts = np.cumsum(counts)
cumulative_percentages = (cumulative_counts / total_ementas) * 100

for i, p in enumerate(hist_plot.patches):
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    # Only annotate if there's a bar (y > 0)
    if y > 0:
        percentage = '{:.1f}%'.format(cumulative_percentages[i])
        ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=9, color='black')

plt.show()

In [ ]:
print("Estatísticas descritivas para 'ementa_word_count':")
display(decisoes_df['ementa_word_count'].describe())

In [ ]:
# Calcula os percentis 5% e 95% para 'ementa_word_count'
lower_bound = decisoes_df['ementa_word_count'].quantile(0.05)
upper_bound = decisoes_df['ementa_word_count'].quantile(0.95)

print(f"Limites de poda (5º e 95º percentil): {lower_bound} a {upper_bound} palavras")

# Número de ementas antes da poda
num_ementas_antes = len(decisoes_df)
print(f"Número de ementas antes da poda: {num_ementas_antes}")

# Aplica a poda baseada nos percentis
decisoes_df = decisoes_df[
    (decisoes_df['ementa_word_count'] >= lower_bound) &
    (decisoes_df['ementa_word_count'] <= upper_bound)
]

# Resetar o índice para garantir alinhamento com a matriz TF-IDF
decisoes_df.reset_index(drop=True, inplace=True)

print(f"Número de ementas após a poda: {len(decisoes_df)}")

#ML

##Stems e Lemmas


In [ ]:
import spacy
from spacy.lang.pt.stop_words import STOP_WORDS

# Tenta usar a GPU se disponível. Deve ser chamado antes de carregar o modelo.
if spacy.prefer_gpu():
    print("SpaCy está usando GPU.")
else:
    print("SpaCy está usando CPU.")

# Carrega o modelo spaCy para português
nlp = spacy.load('pt_core_news_lg')

# Aplica a lematização à coluna 'ementa_clean' utilizando nlp.pipe para paralelização e GPU
# Removendo n_process=-1 para evitar conflitos com GPU
# batch_size ajuda na utilização eficiente da GPU ou CPU
decisoes_df['ementa_lemma'] = [
    ' '.join([
        token.lemma_.lower() for token in doc
        if not token.is_stop
        and not token.is_punct
    ])
    for doc in nlp.pipe(decisoes_df['ementa_clean'], batch_size=2000)
]

In [ ]:
decisoes_df.to_csv('decisoes_df.csv', index=False, encoding='utf-8')

In [ ]:
# Adiciona contagem de lemmas da coluna 'ementa_lemma' em decisoes_df na coluna 'ementa_lemma_count'
decisoes_df['ementa_lemma_count'] = decisoes_df['ementa_lemma'].apply(lambda x: len(x.split(' ')) if isinstance(x, str) else 0)

print("Estatísticas descritivas para 'ementa_lemma_count':")
display(decisoes_df['ementa_lemma_count'].describe())

##BAG OF WORDS (1n-gram)


In [ ]:
#Uso do CountVectorizer para criar uma BoW

CV_lemma = CountVectorizer()
matriz_contagens_lemma = CV_lemma.fit_transform(decisoes_df['ementa_lemma'])


In [ ]:
print("\nTop 20 Lemmas e suas frequências:")
# 1. Extraindo as features com get_feature_names() e convertendo para NumPy explicitamente
df_lemma_dict = pd.DataFrame(CV_lemma.get_feature_names().to_numpy(), columns=['lemma'])

# 2. Somando a matriz e trazendo para a CPU
soma_matriz = matriz_contagens_lemma.sum(axis=0)
if hasattr(soma_matriz, 'get'):
    soma_matriz = soma_matriz.get() # Transfere da GPU (CuPy) para CPU (NumPy)
else:
    try:
        soma_matriz = soma_matriz.to_numpy() # cudf fallback
    except:
        pass

# 3. Adicionando os dados ao DataFrame
df_lemma_dict['vezes_que_aparece'] = np.array(soma_matriz)[0]
df_lemma_dict = df_lemma_dict.sort_values("vezes_que_aparece", ascending=False) # Ordena as linhas do dataframe
pd.set_option('display.max_rows', 20)
display(df_lemma_dict[0:20])


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
# Plotando apenas as 100 primeiras palavras
df_lemma_dict['vezes_que_aparece'][0:20].plot(kind = 'bar')
plt.title('Contagem de palavras - (lemmatizadas)', fontsize = 16, fontweight = 'bold')
# Ajustando as etiquetas para serem apenas as 50 palavras correspondentes
ax.set_xticklabels(df_lemma_dict['lemma'][0:20])
ax.set_xlabel('Palavra', fontsize = 12, fontweight = 'bold')
ax.set_ylabel('Qtde. de vezes que aparece', fontsize = 12, fontweight = 'bold')
plt.show()

##BAG OF WORDS (2n-gram)


In [ ]:
#Uso do CountVectorizer para criar uma BoW


CV_lemma_2 = CountVectorizer(ngram_range=(2,2))
matriz_contagens_lemma = CV_lemma_2.fit_transform(decisoes_df['ementa_lemma'])


In [ ]:


print("\nTop 20 Lemmas e suas frequências:")
df_lemma_dict_2 = pd.DataFrame(CV_lemma_2.get_feature_names().to_numpy(), columns = ['lemma'])
df_lemma_dict_2['vezes_que_aparece'] = matriz_contagens_lemma.sum(axis = 0).tolist()[0]
df_lemma_dict_2 = df_lemma_dict_2.sort_values("vezes_que_aparece", ascending = False) # Ordena as linhas do dataframe
pd.set_option('display.max_rows', 20)
display(df_lemma_dict_2[0:20])

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
# Plotando apenas as 100 primeiras palavras
df_lemma_dict_2['vezes_que_aparece'][0:20].plot(kind = 'bar')
plt.title('Contagem de palavras - (lemmatizadas)', fontsize = 16, fontweight = 'bold')
# Ajustando as etiquetas para serem apenas as 50 palavras correspondentes
ax.set_xticklabels(df_lemma_dict_2['lemma'][0:20])
ax.set_xlabel('Palavra', fontsize = 12, fontweight = 'bold')
ax.set_ylabel('Qtde. de vezes que aparece', fontsize = 12, fontweight = 'bold')
plt.show()

## Vetorização TF-IDF

In [ ]:
# Cria uma instância do TfidfVectorizer para texto baseado em lemas

#1-GRAM
###########

# Converte o conjunto de STOP_WORDS para uma lista para compatibilidade com TfidfVectorizer
vectorizer_lemma_1 = TfidfVectorizer(ngram_range=(1, 1), stop_words=list(STOP_WORDS), max_df=0.95, min_df=10)

# Aplica o vetorizador à coluna 'ementa_lemma'
matriz_tfidf_ementa_lemma_1 = vectorizer_lemma_1.fit_transform(decisoes_df['ementa_lemma'])

# Converte a matriz TF-IDF resultante em um DataFrame
df_matriz_tfidf_ementa_lemma_1 = pd.DataFrame(matriz_tfidf_ementa_lemma_1.toarray().get(), columns = vectorizer_lemma_1.get_feature_names().to_numpy())
pd.set_option('display.max_rows', 10)

################################################################################################
#2-GRAM
# Converte o conjunto de STOP_WORDS para uma lista para compatibilidade com TfidfVectorizer
vectorizer_lemma_2 = TfidfVectorizer(ngram_range=(2, 2), stop_words=list(STOP_WORDS), max_df=0.95, min_df=10)

# Aplica o vetorizador à coluna 'ementa_lemma'
matriz_tfidf_ementa_lemma_2 = vectorizer_lemma_2.fit_transform(decisoes_df['ementa_lemma'])

# Converte a matriz TF-IDF resultante em um DataFrame
df_matriz_tfidf_ementa_lemma_2 = pd.DataFrame(matriz_tfidf_ementa_lemma_2.toarray().get(), columns = vectorizer_lemma_2.get_feature_names().to_numpy())


#1,2-GRAM
# Converte o conjunto de STOP_WORDS para uma lista para compatibilidade com TfidfVectorizer
vectorizer_lemma_12 = TfidfVectorizer(ngram_range=(1, 2), stop_words=list(STOP_WORDS),max_df=0.95, min_df=10)

# Aplica o vetorizador à coluna 'ementa_lemma'
matriz_tfidf_ementa_lemma_12 = vectorizer_lemma_12.fit_transform(decisoes_df['ementa_lemma'])

# Converte a matriz TF-IDF resultante em um DataFrame
df_matriz_tfidf_ementa_lemma_12 = pd.DataFrame(matriz_tfidf_ementa_lemma_12.toarray().get(), columns = vectorizer_lemma_12.get_feature_names().to_numpy())


In [ ]:
def calcular_esparsidade(matriz, nome_matriz):
    # Quantas "células" a matriz tem no total? (Linhas x Colunas)
    total_elementos = matriz.shape[0] * matriz.shape[1]

    # Quantas células têm algum número diferente de zero?
    elementos_nao_zero = matriz.nnz

    # A esparsidade é a porcentagem de zeros
    esparsidade = 1.0 - (elementos_nao_zero / total_elementos)

    print(f"--- Análise da Matriz: {nome_matriz} ---")
    print(f"Dimensões (Documentos x Termos): {matriz.shape}")
    print(f"Elementos não-zero (preenchidos): {elementos_nao_zero}")
    print(f"Esparsidade Matemática: {esparsidade * 100:.4f}%\n")

calcular_esparsidade(matriz_tfidf_ementa_lemma_1, "Unigramas (1,1)")
calcular_esparsidade(matriz_tfidf_ementa_lemma_12, "Mista (1,2)")
calcular_esparsidade(matriz_tfidf_ementa_lemma_2, "Bigramas (2,2)")

In [ ]:
#SVD -> ajuste de parâmetros
from cuml.decomposition import TruncatedSVD
import numpy as np
from kneed import KneeLocator
import matplotlib.pyplot as plt

# 1. O seu código atual que calcula a variância
# Mantive o random_state=1 e n_components=1000 do seu notebook
svd = TruncatedSVD(n_components=1000, random_state=1)

# CORREÇÃO: Convertendo a matriz esparsa para densa para compatibilidade com o cuML SVD
matriz_densa = matriz_tfidf_ementa_lemma_12.toarray()

svd.fit(matriz_densa)
variancia_acumulada = np.cumsum(svd.explained_variance_ratio_.get()) # .get() para trazer para CPU
componentes_x = range(1, len(variancia_acumulada) + 1)

# Algoritmo Kneedle
kneedle = KneeLocator(
    x=list(componentes_x),
    y=variancia_acumulada,
    S=1.0, # Nível de sensibilidade padrão
    curve="concave",
    direction="increasing"
)

# 3.
ponto_de_corte_ideal = kneedle.knee
print(f"O algoritmo detectou o ponto de inflexão exato em: {ponto_de_corte_ideal} componentes.")

# 4. Plotando o gráfico com as referências
plt.figure(figsize=(10, 6))
plt.plot(componentes_x, variancia_acumulada, linewidth=2, color='b', label='Variância Acumulada')
plt.axvline(x=ponto_de_corte_ideal, color='k', linestyle='--', label=f'Ponto de Inflexão (Knee): {ponto_de_corte_ideal}')
plt.axhline(y=0.80, color='r', linestyle='--', label='80% de Variância Explicada') # Linha de referência

plt.title("Detecção Automática do Ponto de Inflexão (Algoritmo Kneedle)")
plt.xlabel("Número de Componentes (Dimensões)")
plt.ylabel("Variância Explicada Acumulada")
plt.grid(True, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
from cuml.decomposition import TruncatedSVD
from cuml.preprocessing import Normalizer
from cuml.pipeline import make_pipeline

# 1. Configura o SVD com o número mágico encontrado pelo Kneedle
# Ajustado para 250 conforme detectado na célula anterior
svd = TruncatedSVD(n_components=250, random_state=1)

# 2. Configura o Normalizador (L2 é o padrão)
normalizador = Normalizer(copy=False)

# 3. Cria uma esteira (Pipeline) para rodar as duas coisas juntas
modelo_lsa = make_pipeline(svd, normalizador)

# 4. Transforma a sua matriz TF-IDF (1,2) na matriz final
# CORREÇÃO: Convertendo para array denso para compatibilidade com cuML
matriz_lsa = modelo_lsa.fit_transform(matriz_tfidf_ementa_lemma_12.toarray())

print(f"Dimensão da matriz original: {matriz_tfidf_ementa_lemma_12.shape}")
print(f"Dimensão da matriz LSA final: {matriz_lsa.shape}")

In [ ]:
!pip install gensim

In [ ]:
import numpy as np
from kneed import KneeLocator
import matplotlib.pyplot as plt
from cuml.decomposition import TruncatedSVD
from cuml.preprocessing import Normalizer
from cuml.pipeline import make_pipeline

def processar_lsa_kneedle(matriz, nome):
    print(f"\n{'='*40}")
    print(f"--- Processando: {nome} ---")
    print(f"{'='*40}")

    # CORREÇÃO: Converter para denso para compatibilidade com cuML
    print("Convertendo matriz para formato denso...")
    matriz_densa = matriz.toarray()

    # 1. Busca do número ideal de componentes (Kneedle)
    print("1. Calculando SVD inicial (1000 componentes). Isso pode demorar um pouco...")
    svd_teste = TruncatedSVD(n_components=1000, random_state=42)
    svd_teste.fit(matriz_densa)

    variancia_acumulada = np.cumsum(svd_teste.explained_variance_ratio_.get()) # .get() para CPU
    componentes_x = range(1, len(variancia_acumulada) + 1)

    kneedle = KneeLocator(
        x=list(componentes_x),
        y=variancia_acumulada,
        S=1.0,
        curve="concave",
        direction="increasing"
    )

    ponto_ideal = kneedle.knee
    print(f"-> Ponto de inflexão (Knee) detectado em: {ponto_ideal} componentes.")

    # Plot do gráfico
    plt.figure(figsize=(8, 4))
    plt.plot(componentes_x, variancia_acumulada, linewidth=2, color='b', label='Variância Acumulada')
    plt.axvline(x=ponto_ideal, color='k', linestyle='--', label=f'Knee: {ponto_ideal}')
    plt.axhline(y=0.80, color='r', linestyle='--', label='80% Variância')
    plt.title(f"Scree Plot - {nome}")
    plt.xlabel("Número de Componentes")
    plt.ylabel("Variância Explicada Acumulada")
    plt.grid(True, alpha=0.5)
    plt.legend()
    plt.show()

    # 2. Pipeline LSA final
    print(f"2. Aplicando LSA final com {ponto_ideal} componentes...")
    svd_final = TruncatedSVD(n_components=ponto_ideal, random_state=42)
    normalizador = Normalizer(copy=False)
    modelo_lsa = make_pipeline(svd_final, normalizador)

    matriz_lsa = modelo_lsa.fit_transform(matriz_densa)
    print(f"-> Dimensão original: {matriz.shape}")
    print(f"-> Dimensão LSA final: {matriz_lsa.shape}\n")

    return matriz_lsa

# Aplicando a função para as matrizes (1,1) e (2,2)
matriz_lsa_1 = processar_lsa_kneedle(matriz_tfidf_ementa_lemma_1, "Unigramas (1,1)")
matriz_lsa_2 = processar_lsa_kneedle(matriz_tfidf_ementa_lemma_2, "Bigramas (2,2)")

In [ ]:
def agrupar_e_avaliar(matriz_tfidf, vectorizer, n_clusters, nome_modelo, n_components_svd, textos_originais):
    print(f"\n{'='*50}\n--- Processando K-Means para: {nome_modelo} ---\n{'='*50}")

    # CORREÇÃO: Converter matriz esparsa para densa para compatibilidade com cuML
    print("Convertendo matriz TF-IDF para formato denso...")
    matriz_densa = matriz_tfidf.toarray()

    # 1. Recria o SVD para poder fazer o inverse_transform
    svd = TruncatedSVD(n_components=n_components_svd, random_state=42)
    normalizador = Normalizer(copy=False)
    modelo_lsa = make_pipeline(svd, normalizador)
    matriz_lsa_reconstruida = modelo_lsa.fit_transform(matriz_densa)

    # 2. Roda K-Means
    print("Treinando K-Means...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    kmeans.fit(matriz_lsa_reconstruida)

    # 3. Extrai as top palavras por cluster
    print("Extraindo Top Palavras...")
    centroides_originais = svd.inverse_transform(kmeans.cluster_centers_).get()

    # cuML usa get_feature_names() em vez de get_feature_names_out()
    try:
        termos = vectorizer.get_feature_names_out()
    except AttributeError:
        termos = vectorizer.get_feature_names().to_numpy()

    top_words_por_cluster = []
    for i, centroide in enumerate(centroides_originais):
        top_indices = centroide.argsort()[::-1][:10]
        # Garantimos que cada termo (mesmo bigrama) seja tratado como uma única string/token
        top_words = [str(termos[ind]) for ind in top_indices]
        top_words_por_cluster.append(top_words)
        print(f"Cluster {i}: {', '.join(top_words)}")

    # 4. Prepara textos para o Gensim com suporte a Bigramas
    print("\nCalculando Coherence Score (C_V)...")
    ngram_range = vectorizer.ngram_range

    def tokenize_ngrams(text, n_range):
        words = str(text).split()
        if n_range == (1, 1):
            return words
        elif n_range == (2, 2):
            # Transforma "palavra1 palavra2" em um único token "palavra1 palavra2"
            return [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
        else: # (1, 2)
            unigrams = words
            bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
            return unigrams + bigrams

    textos_tokenizados = [tokenize_ngrams(t, ngram_range) for t in textos_originais]
    dicionario_gensim = Dictionary(textos_tokenizados)

    # 5. Calcula C_V Coherence
    # Forçamos top_words_por_cluster a ser uma lista de listas de strings
    cm = CoherenceModel(
        topics=top_words_por_cluster,
        texts=textos_tokenizados,
        dictionary=dicionario_gensim,
        coherence='c_v'
    )
    coherence = cm.get_coherence()
    print(f"-> Coherence Score (C_V) para {nome_modelo}: {coherence:.4f}\n")

    return kmeans, top_words_por_cluster, coherence

In [ ]:
# Definições globais de suporte
n_comp_unigrama = matriz_lsa_1.shape[1]
n_comp_bigrama = matriz_lsa_2.shape[1]
n_comp_misto = matriz_lsa.shape[1]
textos = decisoes_df['ementa_lemma'].tolist()

# ==========================================
# PARTE 1: SET UNIGRAMAS (1,1)
# ==========================================
kmeans_1, top_words_1, cv_1 = agrupar_e_avaliar(
    matriz_tfidf=matriz_tfidf_ementa_lemma_1,
    vectorizer=vectorizer_lemma_1,
    n_clusters=10,
    nome_modelo="Unigramas (1,1)",
    n_components_svd=n_comp_unigrama,
    textos_originais=textos
)

res_uni = {'modelo_kmeans': kmeans_1, 'top_words': top_words_1, 'cv_score': cv_1}
joblib.dump(res_uni, 'resultados_kmeans_unigramas.pkl')
print("Resultados do Unigrama salvos em 'resultados_kmeans_unigramas.pkl'")

In [ ]:
# ==========================================
# PARTE 2: SET BIGRAMAS (2,2)
# ==========================================
kmeans_2, top_words_2, cv_2 = agrupar_e_avaliar(
    matriz_tfidf=matriz_tfidf_ementa_lemma_2,
    vectorizer=vectorizer_lemma_2,
    n_clusters=10,
    nome_modelo="Bigramas (2,2)",
    n_components_svd=n_comp_bigrama,
    textos_originais=textos
)

res_bi = {'modelo_kmeans': kmeans_2, 'top_words': top_words_2, 'cv_score': cv_2}
joblib.dump(res_bi, 'resultados_kmeans_bigramas.pkl')
print("Resultados do Bigrama salvos em 'resultados_kmeans_bigramas.pkl'")

In [ ]:
# ==========================================
# PARTE 3: SET MISTO (1,2)
# ==========================================
kmeans_12, top_words_12, cv_12 = agrupar_e_avaliar(
    matriz_tfidf=matriz_tfidf_ementa_lemma_12,
    vectorizer=vectorizer_lemma_12,
    n_clusters=10,
    nome_modelo="Mista (1,2)",
    n_components_svd=n_comp_misto,
    textos_originais=textos
)

res_misto = {'modelo_kmeans': kmeans_12, 'top_words': top_words_12, 'cv_score': cv_12}
joblib.dump(res_misto, 'resultados_kmeans_misto.pkl')
print("Resultados Mistos salvos em 'resultados_kmeans_misto.pkl'")